# Model_KD_Combined

**Knowledge Distillation — 1 Teacher, 2 Students, 2 Init Strategies**

| | Teacher | Student A | Student B |
|---|---|---|---|
| Architecture | `VGG_Pretrained` | `VWW_MobileNetV2` | `VWW_MobileNetV3` |
| Checkpoint | `vgg_pretrained_seed_85.pth` | `mobilenetv2_seed_74.pth` | `mobilenetv3_seed_74.pth` |

**Four training runs produced:**

| Run | Student | Init | Output checkpoint |
|---|---|---|---|
| 1 | MobileNetV2 | FT (warm-start) | `mobilenetv2_kd_ft.pth` |
| 2 | MobileNetV2 | Scratch | `mobilenetv2_kd_scratch.pth` |
| 3 | MobileNetV3 | FT (warm-start) | `mobilenetv3_kd_ft.pth` |
| 4 | MobileNetV3 | Scratch | `mobilenetv3_kd_scratch.pth` |

**Prerequisites:** Run `Model_VGG_Pretrained`, `Model_MobileNetV2`, `Model_MobileNetV3` first.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time
import torch

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VGG_Pretrained, VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate, train_kd

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# ── Checkpoints ──────────────────────────────────────────────────────
TEACHER_CKPT   = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"
MV2_CKPT       = f"{SAVE_DIR}/mobilenetv2_seed_74.pth"
MV3_CKPT       = f"{SAVE_DIR}/mobilenetv3_seed_74.pth"

# ── Output paths ─────────────────────────────────────────────────────
MV2_KD_FT_PATH     = f"{SAVE_DIR}/mobilenetv2_kd_ft.pth"
MV2_KD_SCR_PATH    = f"{SAVE_DIR}/mobilenetv2_kd_scratch.pth"
MV3_KD_FT_PATH     = f"{SAVE_DIR}/mobilenetv3_kd_ft.pth"
MV3_KD_SCR_PATH    = f"{SAVE_DIR}/mobilenetv3_kd_scratch.pth"

# ── KD hyperparameters ────────────────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 30
KD_LR_FT      = 3e-4   # lower LR for warm-start: preserve existing weights
KD_LR_SCR     = 1e-3   # standard LR for scratch
KD_WD         = 1e-4
KD_PATIENCE   = 10

In [ ]:
# ── Load teacher (frozen for all runs) ──────────────────────────────
teacher = VGG_Pretrained().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_acc = evaluate(teacher, val_loader, device)
print(f"✅ Teacher loaded  |  Val acc: {teacher_acc*100:.2f}%")

---
## Run 1 — MobileNetV2 · KD Fine-Tune
Warm-start from the `mobilenetv2_seed_74` baseline checkpoint, then distil.

In [ ]:
# ── MobileNetV2 · KD FT ─────────────────────────────────────────────
student_mv2_ft = VWW_MobileNetV2().to(device)
student_mv2_ft.load_state_dict(torch.load(MV2_CKPT, map_location=device))
baseline_mv2   = evaluate(student_mv2_ft, val_loader, device)
print(f"MobileNetV2 baseline : {baseline_mv2*100:.2f}%")
if teacher_acc <= baseline_mv2:
    print("⚠️  Teacher not stronger than student — KD may not help")

mv2_ft_acc, mv2_ft_time = train_kd(
    student      = student_mv2_ft,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_FT,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV2_KD_FT_PATH,
)

---
## Run 2 — MobileNetV2 · KD Scratch
Random initialisation — teacher supervision only, no pre-trained weights.

In [ ]:
# ── MobileNetV2 · KD Scratch ─────────────────────────────────────────
student_mv2_scr = VWW_MobileNetV2().to(device)
print("✅ MobileNetV2 initialised from scratch")

mv2_scr_acc, mv2_scr_time = train_kd(
    student      = student_mv2_scr,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_SCR,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV2_KD_SCR_PATH,
)

---
## Run 3 — MobileNetV3 · KD Fine-Tune
Warm-start from the `mobilenetv3_seed_74` baseline checkpoint, then distil.

In [ ]:
# ── MobileNetV3 · KD FT ─────────────────────────────────────────────
student_mv3_ft = VWW_MobileNetV3().to(device)
student_mv3_ft.load_state_dict(torch.load(MV3_CKPT, map_location=device))
baseline_mv3   = evaluate(student_mv3_ft, val_loader, device)
print(f"MobileNetV3 baseline : {baseline_mv3*100:.2f}%")
if teacher_acc <= baseline_mv3:
    print("⚠️  Teacher not stronger than student — KD may not help")

mv3_ft_acc, mv3_ft_time = train_kd(
    student      = student_mv3_ft,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_FT,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV3_KD_FT_PATH,
)

---
## Run 4 — MobileNetV3 · KD Scratch
Random initialisation — teacher supervision only, no pre-trained weights.

In [ ]:
# ── MobileNetV3 · KD Scratch ─────────────────────────────────────────
student_mv3_scr = VWW_MobileNetV3().to(device)
print("✅ MobileNetV3 initialised from scratch")

mv3_scr_acc, mv3_scr_time = train_kd(
    student      = student_mv3_scr,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR_SCR,
    weight_decay = KD_WD,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = KD_PATIENCE,
    save_path    = MV3_KD_SCR_PATH,
)

---
## Results Summary

In [ ]:
# ── Load best checkpoints and evaluate ─────────────────────────────
def reload_eval(model_cls, ckpt):
    m = model_cls().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return evaluate(m, val_loader, device)

final_mv2_ft  = reload_eval(VWW_MobileNetV2, MV2_KD_FT_PATH)
final_mv2_scr = reload_eval(VWW_MobileNetV2, MV2_KD_SCR_PATH)
final_mv3_ft  = reload_eval(VWW_MobileNetV3, MV3_KD_FT_PATH)
final_mv3_scr = reload_eval(VWW_MobileNetV3, MV3_KD_SCR_PATH)

print("=" * 62)
print(f"{'Run':<30} {'Val Acc':>10} {'Time (min)':>12}")
print("-" * 62)
print(f"Teacher (VGG_Pretrained)       {teacher_acc*100:>9.2f}%            —")
print("-" * 62)
print(f"MobileNetV2 baseline           {baseline_mv2*100:>9.2f}%            —")
print(f"MobileNetV2  KD FT             {final_mv2_ft*100:>9.2f}% {mv2_ft_time:>11.1f}")
print(f"MobileNetV2  KD Scratch        {final_mv2_scr*100:>9.2f}% {mv2_scr_time:>11.1f}")
print("-" * 62)
print(f"MobileNetV3 baseline           {baseline_mv3*100:>9.2f}%            —")
print(f"MobileNetV3  KD FT             {final_mv3_ft*100:>9.2f}% {mv3_ft_time:>11.1f}")
print(f"MobileNetV3  KD Scratch        {final_mv3_scr*100:>9.2f}% {mv3_scr_time:>11.1f}")
print("=" * 62)

# Delta over baseline
print()
print("Δ over baseline:")
print(f"  MobileNetV2  FT     : {(final_mv2_ft  - baseline_mv2)*100:+.2f}%")
print(f"  MobileNetV2  Scratch: {(final_mv2_scr - baseline_mv2)*100:+.2f}%")
print(f"  MobileNetV3  FT     : {(final_mv3_ft  - baseline_mv3)*100:+.2f}%")
print(f"  MobileNetV3  Scratch: {(final_mv3_scr - baseline_mv3)*100:+.2f}%")